In [1]:
import pandas as pd
import numpy as np

RAW = "../data/raw/"

# Load raw files
orders      = pd.read_csv(RAW + "olist_orders_dataset.csv")
items       = pd.read_csv(RAW + "olist_order_items_dataset.csv")
payments    = pd.read_csv(RAW + "olist_order_payments_dataset.csv")
reviews     = pd.read_csv(RAW + "olist_order_reviews_dataset.csv")
customers   = pd.read_csv(RAW + "olist_customers_dataset.csv")
products    = pd.read_csv(RAW + "olist_products_dataset.csv")
translation = pd.read_csv(RAW + "product_category_name_translation.csv")

# ── Re-apply Day 4 cleaning ──────────────────────────────────────────

# Convert date columns
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

reviews['review_creation_date']    = pd.to_datetime(reviews['review_creation_date'],    errors='coerce')
reviews['review_answer_timestamp'] = pd.to_datetime(reviews['review_answer_timestamp'], errors='coerce')
items['shipping_limit_date']       = pd.to_datetime(items['shipping_limit_date'],        errors='coerce')

# Filter to delivered orders only
orders = orders[orders['order_status'] == 'delivered'].copy()
orders = orders.dropna(subset=['order_delivered_customer_date'])

# Deduplicate reviews — keep latest per order
reviews = (reviews
    .sort_values('review_answer_timestamp', ascending=False)
    .drop_duplicates(subset='order_id', keep='first')
    .copy()
)

# Fix missing product categories
products['product_category_name'] = products['product_category_name'].fillna('unknown')

print(f"orders (delivered, clean): {len(orders):,}")
print(f"items:                     {len(items):,}")
print(f"payments:                  {len(payments):,}")
print(f"reviews (deduplicated):    {len(reviews):,}")
print(f"customers:                 {len(customers):,}")
print(f"products:                  {len(products):,}")
print(f"translation:               {len(translation):,}")

orders (delivered, clean): 96,470
items:                     112,650
payments:                  103,886
reviews (deduplicated):    98,673
customers:                 99,441
products:                  32,951
translation:               71


In [2]:
# STEP 1: Aggregate payments BEFORE any join
# This is critical — without this, joining payments to items
# would multiply item rows by payment rows

def get_dominant_payment_type(series):
    """Return the most frequent payment type for an order."""
    return series.value_counts().index[0]

payments_agg = (payments
    .groupby('order_id')
    .agg(
        total_payment_value   = ('payment_value',          'sum'),
        payment_installments  = ('payment_installments',   'max'),
        dominant_payment_type = ('payment_type',           get_dominant_payment_type)
    )
    .reset_index()
)

print(f"Payments BEFORE aggregation: {len(payments):,} rows")
print(f"Payments AFTER aggregation:  {len(payments_agg):,} rows")
print(f"\nSample aggregated payments:")
print(payments_agg.head(3))
print(f"\nPayment types after aggregation:")
print(payments_agg['dominant_payment_type'].value_counts())

Payments BEFORE aggregation: 103,886 rows
Payments AFTER aggregation:  99,440 rows

Sample aggregated payments:
                           order_id  total_payment_value  \
0  00010242fe8c5a6d1ba2dd792cb16214                72.19   
1  00018f77f2f0320c557190d7a144bdd3               259.83   
2  000229ec398224ef6ca0657da4fc703e               216.87   

   payment_installments dominant_payment_type  
0                     2           credit_card  
1                     3           credit_card  
2                     5           credit_card  

Payment types after aggregation:
dominant_payment_type
credit_card    75270
boleto         19784
voucher         2856
debit_card      1527
not_defined        3
Name: count, dtype: int64


In [3]:
# STEP 2-5: Build order-level table first
# orders + customers + payments + reviews
# All should be 1-to-1 at this stage

# Join orders + customers
df = orders.merge(customers, on='customer_id', how='left')
print(f"After orders + customers:  {len(df):,} rows | {df['order_id'].nunique():,} unique orders")

# Join + payments (already aggregated)
df = df.merge(payments_agg, on='order_id', how='left')
print(f"After + payments:          {len(df):,} rows | {df['order_id'].nunique():,} unique orders")

# Join + reviews (already deduplicated)
reviews_slim = reviews[['order_id', 'review_score', 'review_answer_timestamp']].copy()
df = df.merge(reviews_slim, on='order_id', how='left')
print(f"After + reviews:           {len(df):,} rows | {df['order_id'].nunique():,} unique orders")

print(f"\n✅ Order-level base table built: {len(df):,} rows")
print("Row count should still match delivered orders — verify above ☝️")

After orders + customers:  96,470 rows | 96,470 unique orders
After + payments:          96,470 rows | 96,470 unique orders
After + reviews:           96,470 rows | 96,470 unique orders

✅ Order-level base table built: 96,470 rows
Row count should still match delivered orders — verify above ☝️


In [4]:
# STEP 6: NOW join items — this expands to order-item level
# This is intentional — we expect more rows than orders

df = df.merge(items, on='order_id', how='left')
print(f"After + items:             {len(df):,} rows | {df['order_id'].nunique():,} unique orders")
print("⚠️  Row count increased — this is expected (order-item level now)")
print(f"Average items per order:   {len(df) / df['order_id'].nunique():.2f}")

After + items:             110,189 rows | 96,470 unique orders
⚠️  Row count increased — this is expected (order-item level now)
Average items per order:   1.14


In [5]:
# STEP 7: Join products
df = df.merge(products[['product_id', 'product_category_name']], on='product_id', how='left')
print(f"After + products:          {len(df):,} rows | {df['order_id'].nunique():,} unique orders")

# STEP 8: Join translation for English category names
df = df.merge(translation, on='product_category_name', how='left')
print(f"After + translation:       {len(df):,} rows | {df['order_id'].nunique():,} unique orders")

# Check how many rows have no English category name
no_translation = df['product_category_name_english'].isnull().sum()
print(f"\nRows missing English category name: {no_translation:,}")
print("(These are 'unknown' category products or 2 untranslated categories)")

# Fill missing English names with 'unknown'
df['product_category_name_english'] = df['product_category_name_english'].fillna('unknown')
print(f"After filling missing translations: {df['product_category_name_english'].isnull().sum()} nulls remaining")

After + products:          110,189 rows | 96,470 unique orders
After + translation:       110,189 rows | 96,470 unique orders

Rows missing English category name: 1,559
(These are 'unknown' category products or 2 untranslated categories)
After filling missing translations: 0 nulls remaining


In [6]:
print("=== FINAL JOIN VERIFICATION ===")
print(f"Total rows in final dataset:      {len(df):,}")
print(f"Unique orders in final dataset:   {df['order_id'].nunique():,}")
print(f"Unique customers:                 {df['customer_id'].nunique():,}")
print(f"Unique products:                  {df['product_id'].nunique():,}")

print("\n=== NULL CHECK ON KEY COLUMNS ===")
key_cols = [
    'order_id', 'customer_id', 'order_purchase_timestamp',
    'order_delivered_customer_date', 'customer_state',
    'price', 'freight_value', 'review_score',
    'dominant_payment_type', 'product_category_name_english'
]
for col in key_cols:
    nulls = df[col].isnull().sum()
    pct   = nulls / len(df) * 100
    status = "✅" if nulls == 0 else "⚠️ "
    print(f"{status} {col:<45} nulls: {nulls:,} ({pct:.1f}%)")

print("\n=== SPOT CHECK — Sample 3 rows ===")
print(df[['order_id', 'customer_state', 'price', 'freight_value',
          'review_score', 'dominant_payment_type',
          'product_category_name_english']].head(3).to_string())

=== FINAL JOIN VERIFICATION ===
Total rows in final dataset:      110,189
Unique orders in final dataset:   96,470
Unique customers:                 96,470
Unique products:                  32,214

=== NULL CHECK ON KEY COLUMNS ===
✅ order_id                                      nulls: 0 (0.0%)
✅ customer_id                                   nulls: 0 (0.0%)
✅ order_purchase_timestamp                      nulls: 0 (0.0%)
✅ order_delivered_customer_date                 nulls: 0 (0.0%)
✅ customer_state                                nulls: 0 (0.0%)
✅ price                                         nulls: 0 (0.0%)
✅ freight_value                                 nulls: 0 (0.0%)
⚠️  review_score                                  nulls: 827 (0.8%)
⚠️  dominant_payment_type                         nulls: 3 (0.0%)
✅ product_category_name_english                 nulls: 0 (0.0%)

=== SPOT CHECK — Sample 3 rows ===
                           order_id customer_state   price  freight_value  review_scor

In [7]:
# Pick one specific order_id and trace it through all tables
# This is the most reliable way to confirm joins are correct

sample_order = df['order_id'].iloc[0]
print(f"Tracing order_id: {sample_order}\n")

print("--- In orders table ---")
print(orders[orders['order_id'] == sample_order][
    ['order_id', 'order_purchase_timestamp', 'order_delivered_customer_date']
].to_string(index=False))

print("\n--- In items table ---")
print(items[items['order_id'] == sample_order][
    ['order_id', 'price', 'freight_value']
].to_string(index=False))

print("\n--- In payments table (raw) ---")
print(payments[payments['order_id'] == sample_order][
    ['order_id', 'payment_type', 'payment_value']
].to_string(index=False))

print("\n--- In final joined dataset ---")
print(df[df['order_id'] == sample_order][
    ['order_id', 'price', 'freight_value',
     'total_payment_value', 'dominant_payment_type', 'review_score']
].to_string(index=False))

Tracing order_id: e481f51cbdc54678b7cc49136f2d6af7

--- In orders table ---
                        order_id order_purchase_timestamp order_delivered_customer_date
e481f51cbdc54678b7cc49136f2d6af7      2017-10-02 10:56:33           2017-10-10 21:25:13

--- In items table ---
                        order_id  price  freight_value
e481f51cbdc54678b7cc49136f2d6af7  29.99           8.72

--- In payments table (raw) ---
                        order_id payment_type  payment_value
e481f51cbdc54678b7cc49136f2d6af7  credit_card          18.12
e481f51cbdc54678b7cc49136f2d6af7      voucher           2.00
e481f51cbdc54678b7cc49136f2d6af7      voucher          18.59

--- In final joined dataset ---
                        order_id  price  freight_value  total_payment_value dominant_payment_type  review_score
e481f51cbdc54678b7cc49136f2d6af7  29.99           8.72                38.71               voucher           4.0


In [8]:
# Save the joined dataset for use in Day 6 (feature engineering)
output_path = "../data/cleaned/ecommerce_joined.csv"
df.to_csv(output_path, index=False)

print(f"✅ Joined dataset saved to: {output_path}")
print(f"   Rows:    {len(df):,}")
print(f"   Columns: {df.shape[1]}")
print(f"\nColumn list:")
for col in df.columns:
    print(f"  - {col}")

✅ Joined dataset saved to: ../data/cleaned/ecommerce_joined.csv
   Rows:    110,189
   Columns: 25

Column list:
  - order_id
  - customer_id
  - order_status
  - order_purchase_timestamp
  - order_approved_at
  - order_delivered_carrier_date
  - order_delivered_customer_date
  - order_estimated_delivery_date
  - customer_unique_id
  - customer_zip_code_prefix
  - customer_city
  - customer_state
  - total_payment_value
  - payment_installments
  - dominant_payment_type
  - review_score
  - review_answer_timestamp
  - order_item_id
  - product_id
  - seller_id
  - shipping_limit_date
  - price
  - freight_value
  - product_category_name
  - product_category_name_english
